In [ ]:
%pip install pandas mlxtend matplotlib seaborn networkx seaborn plotly

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import seaborn as sns
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "iframe"

# Association Rules
## The data in for this assignment was generated using the method by Agrawal and Srikant (random.patterns) to simulate transactions (random.transactions) which contains correlated items.  10,000 transactions occurred with 100 items to choose from.  The average length of the transactions is 10 items.  Note: You will need to load the arules and arulesViz R packages to complete this assignment.

- Import the ```AssociationRules.csv``` transaction data file.

### Хотя формат файла csv, на деле он не удовлетворяет формату в полной мере - нет названий столбцов, разное количество элементов в каждой строке, отсутствует логическая структура распределения элементов по строкам и столбцам - в следствие чего буду рассматривать его как обычное перечисление товаров через пробел/'\n'

In [ ]:
transactions = [
    line.strip().split()
    for line in open("resources/3-lab/AssociationRules.csv")
]
transactions

# Create a frequent item plot, and a frequent item table. 


In [ ]:
item_frequent = pd.Series([i for r in transactions for i in r]).value_counts().reset_index()
item_frequent.columns = ["item_number", "count"]

item_frequent_sorted = item_frequent.sort_values(by="count", ascending=True)

plt.figure(figsize=(10, 22))
plt.barh(item_frequent_sorted["item_number"], item_frequent_sorted["count"], color="#1abc9c")
plt.xlabel("Frequent", fontsize=12)
plt.ylabel("Item", fontsize=12)
plt.title("Frequent item plot", fontsize=14)
plt.show()

## a.      Determine the most frequent item bought in the store.

In [ ]:
item_frequent[item_frequent['count'] == item_frequent['count'].max()]

## b.      How many items were bought in the largest transaction?

In [ ]:
max([len(r) for r in transactions])

#  Mine the Association rules with a minimum Support of 1% and a minimum Confidence of 0%.
- ```поддержка``` - доля транзакций, в которых одновременно встречается товар X_i и X_p делить на количество транзакций (Популярность набора)
 $$Supp(X_i \implies X_p) = \frac{\text{Количество транзакций с } X_i \text{ и } X_p}{\text{Общее число транзакций } N}$$
- ```Уверенность``` - Условная вероятность "Если есть X_i, то есть и X_p"
   $$Conf(X_i \implies X_p) = \frac{Supp(X_i \cup X_p)}{Supp(X_i)} = \frac{\text{Количество транзакций с } X_i \text{ и } X_p}{\text{Количество транзакций с } X_i}$$
### Алгоритм ```Apriori```
Помогает найти частые наборы товаров, работает на своействе моотонности: если набор товаров редкий (То есть поддержка данного товара ниже порогового значения), то любое множество Х, которому принадлежит данный набор товаров также будет иметь поддержку ниже порогового значения, в следствие чего такие наборы будут отсекаться как "априори неподходящие"
- ```TransactionEncoder.fit()``` - принимает итерариуеымй объект, содержащий итерарируемые объекты, затем каждому элементу присваивает уникальный индекс, строя словарь ```{индекс: элемент}```
- ```TransactionEncoder.transform()``` - Смотрит словарь экземпляра ```TransactionEncoder```, после чего создает для каждой строки массив из нулей, чей размер равен количеству различных элементов, после чего в нужных колонках меняет ```0``` на ```1```
- ```apriori()``` - принимает матрицу one-hot encoding и порог поддержки, находит все наборы товаров, которые встречаются чаще порога поддержки
- ```association_rules()``` - принимает на вход найденные частые наборы, тип метрики (```confidence```, ```support```, ```lift```, ```leverage```, ```conviction```, ```zhangs_metric```) и порог уверенности, после чего генерирует логические правила вида "Если X_i, то X_p" и рассчитывает для каждого правила все статистические метрики, возвращает данные в виде ```DataFrame```
### Структура возвращённого ```association_rules()``` объекта 
| antecedents | consequents | antecedent support | consequent support | support | confidence | lift | representativity | leverage | conviction | zhangs_metric | jaccard | certainty | kulczynski |
|-|-|-|-|-|-|-|-|-|-|-|-|-|-|
| Неизменяемый набор элементов-условий (Если это есть в наборе, то ...) | Неизменяемый набор элементов следствий (... , то есть и эти элементы) | Поддержка набора элементов-условий | Поддержка набора элементов-следствий | Поддержка правила - доля наборов, содержащих элементы набора, делить на общее количество | Уверенность правила | Показывает, во сколько раз реальная частота покупки набора больше ожидаемой (Покупка товаров независимо друг от друга) ```lift > 1 - положительная связь; lift = 1 - независимы; lift < 1 - отрицательная связь``` | Показывает долю информации датасета, что охвачена набором элементов |разница между реальной частотой и ожидаемой | Степень зависимости следствия от условия зависимости  | Ассоциация Чжана | Коэффициент Жаккарда | Измеряет степень уверенности в правиле с поправкой на частоту появления следствия | Индекс Кульчинского |

In [ ]:
te = TransactionEncoder()
items_te = te.fit(transactions).transform(transactions)
df = pd.DataFrame(items_te, columns=te.columns_)

frequent_itemsets = apriori(df, min_support=0.01, use_colnames=True)

rules = association_rules(
    frequent_itemsets, metric="confidence", min_threshold=0.0
)

rules 

### c.       How many rules appear in the data? 

In [ ]:
len(rules)

### d.      How many rules are observed when the minimum confidence is 50%.

In [ ]:
len(rules[rules['confidence'] >= 0.5])

### e.      Explain how the specified confidence impacts the number of rules. 

Каждое правило обладает процентом уверенности - вероятностью того, что при нахождении элементов-условий в наборе в наборе окажутся и элементы-следствия
С её помощью мы можем определить, как часто то или иное правило оказывается "срабатывающим" (То есть при покупке товаров-условий люди купили и товары-следствия)

Чем меньше порог уверенности правила, тем более неточные правила попадают в выборку

Зависимость порога уверенности и количества правил обратно пропорциональная

##  Create a scatter plot comparing the parameters support and confidence on the axis, and lift with shading.  

In [ ]:
plt.figure(figsize=(10, 10))
scatter = sns.scatterplot(
    data = rules,
    x = 'support', 
    y = 'confidence',
    hue = 'lift',
    size = 'lift',
    palette = 'viridis',
    alpha = 0.8,
    edgecolor = None
)

plt.title("Association rules", fontsize = 14)
plt.xlabel('Support', fontsize = 12)
plt.ylabel('Confidence', fontsize = 12)

plt.tight_layout()

plt.show()

### f.        Identify the positioning of the interesting rules. 

In [ ]:
rules[rules['lift'] > 12]

##  Compare support and lift.  

| Support | Lift |
|:-:|:-:|
| Измеряет частоту встречаемости набора в датасете | Измеряет степень неслучайности встречи набора |
| $P(X \cup Y)$ | $\frac{P(X \cup Y)}{P(X) \cdot P(Y)}$ |
| от 0 до 1 | от 0 до +inf |
| Отсекает непопулярные комбинации | Находит сильные закономерности |

### g.      Create a scatter plot measuring support vs. lift; record your observations.

In [ ]:
plt.figure(figsize=(10, 10))
scatter = sns.scatterplot(
    data = rules,
    x = 'support', 
    y = 'lift',
    hue = 'confidence',
    size = 'confidence',
    palette = 'viridis',
    alpha = 0.8,
    edgecolor = None
)

plt.title("Association rules", fontsize = 14)
plt.xlabel('Support', fontsize = 12)
plt.ylabel('Lift', fontsize = 12)

plt.tight_layout()

plt.show()

### h.      Where are the rules located that would be considered interesting and useful? 

In [ ]:
rules[(rules['lift'] > 12) | (rules['support'] > 0.15)]

### i.        One downside to the Apriori algorithm, is that extraneous rules can be generated that are not particularly useful.  Identify where these rules are located on the graph.  Explain the relationship between the expected observation of these itemsets and the actual observation of the itemsets. 
Снизу слева, они непопулярны и слабые

## j.        Using the interaction tool for a scatter plot, identify 3 rules that appear in at least 10% of the transactions by coincidence.  

In [ ]:
rules[rules['support'] >= 0.1].assign(diff=(rules['lift'] - 1).abs()).sort_values('diff').head(3)

##  Identify the most interesting rules by extracting the rules in which the Confidence is >0.8.   Observe the output of the data table for the most interesting rules. 

In [ ]:
rules[rules['confidence'] >= 0.8].sort_values(by='lift', ascending=False)

### k.       Sort the rules stating the highest lift first.  Provide the 10 rules with the lowest lift. Do they appear to be coincidental?  Why or why not? 

Да, они получаются случайно, поскольку lift является отношением "количества таких наборов в датесете" и "количества таких наборов при условии, что каждый набор генерируется независимо", если lift <= 1, то это означает, что частота появления наборов меньше "частоты появления при равновероятностном распределении наборов"

In [ ]:
rules.sort_values(by='lift', ascending=False).tail(10)

##   Create a Matrix-based visualization of two measures with colored squares.  The two measures should compare confidence and lift (have recorded = FALSE).  Note that 4 interesting rules stand out on the graph. 

In [ ]:
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

fig = px.scatter(
    rules,
    x='consequents_str',
    y='antecedents_str',
    color='lift',
    size='confidence',
    color_continuous_scale = 'Blues',
    hover_data=['support', 'confidence', 'lift'], # Какие данные вывести при наведении на квадрат
    title="Matrix-based visualization"
)

fig.update_traces(marker=dict(symbol='square', line=dict(width=0)))

fig.update_layout(
    xaxis_tickangle=-45,
    xaxis_title="Consequents",
    yaxis_title="Antecedents",
)

fig.show()

### l.        Identify these rules and explain their appearance.

Они могли появиться по причине
- товары antecedents нуждаются в товарах consequents (Например, Телеофон + зарядка)
- Продаются в формате акции (Возьми antecedents и получи скидку 50% на consequents)
- Товары часто используются в быту в связке (Хлеб, сыр, колбаса)

In [ ]:
rules.sort_values(by=['lift', 'confidence'], ascending=False).head(4)

### m.    What can you infer about rules represented by a dark blue color?
- У них высокий lift

##   Extract the three rules with the highest lift.

In [ ]:
three_rules = (rules.sort_values(by='lift', ascending=False).iloc[0:5]).drop(index=[21348, 21346])

three_rules

### n.      Record the Rules.  Explain why these rules vary from the rules in Step 3. 
??? Что именно подразумеваеися под step 3 ??? 
### o.      Create a Graph-based visualization with items and rules as vertices.

In [ ]:
G = nx.DiGraph()

for i, r in three_rules.reset_index(drop=True).iterrows():
    rid = f"Rule {i+1} ({r['lift']:.0f})"
    G.add_edges_from([(item, rid) for item in r['antecedents']] + 
                     [(rid, item) for item in r['consequents']])

plt.figure(figsize=(10, 6))
nx.draw(
    G, 
    with_labels=True,
    node_color=['tomato' if 'Rule' in n else 'skyblue' for n in G.nodes()],
    node_size=2500, 
)
plt.axis('off')
plt.show()

### p.      Based on your observations, explain how you would expect association rules to relate to order (i.e. the number of items contained in the rule).
- Поддержка будет убывать, тк чем больше набор, тем меньше наборов могут выступать ему надмножеством
- Уверенность зачастую будет расти, тк дополнительными условиями мы сужаем выборку => делим на меньшее число => уверенность растет

  ## Create a training set from the first 8,000 transactions. Create a testing set from the last 2,000 transactions.  Run the algorithm on each dataset.  Compare the results.

In [ ]:
train_df = df.loc[:int(len(df)*0.8) - 1]
test_df = df.loc[int(len(df) * 0.8):]
print(len(train_df), len(test_df))

Правила на train и test не совпали, это может быть связано с недостаточным объемом test выборки

In [ ]:
frequent_itemsets_train = apriori(train_df, min_support=0.01, use_colnames=True)
rules_train = association_rules(
    frequent_itemsets_train, metric="confidence", min_threshold=0.0
)

frequent_itemsets_test = apriori(test_df, min_support=0.01, use_colnames=True)
rules_test = association_rules(
    frequent_itemsets_test, metric="confidence", min_threshold=0.0
)

print("Train")
display(rules_train.sort_values(by="lift", ascending=False).head(10))

print("Test")
display(rules_test.sort_values(by="lift", ascending=False).head(10))

### q.      Justify that the relationships we see are not just an artifact of the data we started with.
Мы можем это доказать это с помощью lift, поддержки и уверенности, поскольку они напрямую оценивают насколько неслучайны те или иные правила. Хотя test и train сравнения не дали желаемых результатов, на этом считать данные артефактами нельзя ввиду малого объема данных

### r.       Can we conclude that the association rules we found are actually true in the population we are studying?

Нет, поскольку train и test различаются и не имеют "общих правил" (Опять же, в связи с объемом выборки это не обязательно реджектит правила, но и совсем игнорировать такой маркер странное решение)
С другой стороны, у нас есть правила с очень хорошими lift и уверенностями, в связи с чем можно предположить, что наш датасет является еще и time series и - условно - это отчет за буднюю неделю, где первые 8к это пон-чет, а остальные 2к это пятница, тогда вполне очевидно, что перед выходными люди будут делать выбор в сторону других товаров ( Многие любят "отметить" завершение недели, в связи с чем покупают нетипичные для пон-чет продукты)